# Your Robot's Auto-Distancing Data

Your FTC robot needs to shoot from any distance on the field. Your team measured the motor power needed at 13 different distances. The question:

**How do you predict the right power for a distance you didn't measure?**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Your team's actual measurements
distances = np.array([47, 50, 57, 60, 65, 68, 72, 76, 83, 90, 95, 100, 110])  # distance from target
power = np.array([975, 985, 1000, 1005, 1010, 1012, 1020, 1025, 1031, 1080, 1120, 1150, 1200])  # motor power

## Plot the data

What shape is it? Linear? Curved? Just eyeball it.

In [ ]:
# plt.scatter(x, y) plots points
# plt.xlabel(), plt.ylabel(), plt.title() for labels
# plt.show() to display


## Fit a line (degree 1)

Use `np.polyfit(x, y, degree)` to find the best-fit polynomial coefficients.
Use `np.polyval(coefficients, x)` to evaluate the polynomial at given x values.

Fit a straight line through your data. Plot it on top of the scatter plot. How does it look?

Compute the **mean squared error** (MSE) — the average of `(predicted - actual)²` across all points. This is your measure of how bad the fit is.

In [ ]:
# Fit a degree-1 polynomial (a line)

# Plot the line on top of your data

# Compute MSE


## Try degree 2

Fit a degree-2 polynomial (a parabola). Plot it alongside the line. Compute the MSE.

Better? By how much? Which would you trust more for a distance you haven't tested?

In [ ]:
# Fit degree 2

# Plot both degree 1 and degree 2 on top of the data

# Compare MSE


## Go higher

Fit degree 5. Then degree 10. Then degree 12.

Plot each one. What happens to the MSE?

Tip: to see what the polynomial does *between* your data points, plot it using a smooth set of x values:
```python
x_smooth = np.linspace(distances.min(), distances.max(), 200)
y_smooth = np.polyval(coefficients, x_smooth)
```

In [ ]:
# Fit degree 5, 10, 12

# Plot each one using x_smooth so you can see the curve between data points

# Print MSE for each


## ⚡ Stop and think

The degree-12 polynomial passes through every single data point. Its training error is basically zero. Perfect score.

But look at what it does *between* the data points. Look at what it predicts just beyond the edges of your data.

**Would you put this on your robot?**

You have 13 data points and a model that memorized all of them. It hasn't learned the *pattern* — it's learned the *noise*. This is called **overfitting**.

So if you can't trust zero error... **how do you actually choose the right model?**

Think about this before scrolling down. There's a simple idea.

## One idea: hide some data from yourself

What if you set aside 3-4 data points *before* fitting? Train on the remaining points, then check how well the model predicts the ones it never saw.

A model that memorized the training data will fail on the held-out points. A model that learned the real pattern will do fine.

Try it: randomly pick 3-4 indices to hold out. Fit degrees 1 through 6 on the rest. Compute MSE on *only the held-out points*. Which degree wins?

In [ ]:
# Pick indices to hold out (e.g., np.random.choice)
# Split into train and test
# Fit degrees 1-6 on train, compute MSE on test
# Which degree has the lowest test MSE?


## But the split was random

Run the cell above a few times. You'll get different answers depending on which points you held out.

That's a problem. Your choice of model shouldn't depend on luck.

How would you make this more robust? What if you tried *every possible* split — or at least many of them — and averaged the results?

## Cross-validation

That's exactly what **cross-validation** does. It systematically rotates which points are held out, fits the model each time, and averages the test errors across all splits.

`scikit-learn` has a function for this: `cross_val_score`. It handles the splitting and scoring for you.

📖 [sklearn: Cross-validation — evaluating estimator performance](https://scikit-learn.org/stable/modules/cross_validation.html) — the first section + the flowchart diagram explain the core idea.

Use cross-validation to compare degrees 1 through 6. Which degree consistently gives the lowest error on data it hasn't seen?

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

# For each degree 1-6:
#   model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
#   scores = cross_val_score(model, distances.reshape(-1, 1), power,
#                            cv=4, scoring='neg_mean_squared_error')
#   print the mean score (note: scores are negative MSE — closer to 0 is better)


## The physics agrees

Cross-validation should tell you degree 2 wins. That's not a coincidence — there's a physical reason.

From basic physics, the farther your target, the faster you need to launch — that's roughly a **linear** relationship. But air drag complicates things: drag force grows with the **square** of speed, introducing a quadratic term that dominates at higher velocities.

So physics predicts the relationship should be a second-degree polynomial. And cross-validation — knowing nothing about physics, just looking at prediction error — independently arrived at the same answer.

**The math found the physics.** That's model selection: not just curve fitting, but discovering real structure in data.

## Final fit

Now that you know the best degree, fit it on **all 13 points** (no holdout — you've already chosen your model, now you want the best possible version of it).

Plot it. This is what your robot could use during a match.

In [ ]:
# Fit the winning degree on all data
# Plot it with x_smooth
# Overlay the original data points


## Optional: the calc connection

What you just did — fitting a polynomial by minimizing the sum of squared errors — is an optimization problem.

You're looking for the coefficients that minimize a function (total error). The minimum is where the derivative equals zero. You've done this in Calc BC.

`np.polyfit` solves this analytically (using linear algebra). But there's another way: start with random coefficients, compute the error, compute the gradient (which direction makes the error smaller?), take a small step, repeat. That's called **gradient descent**.

It's slower than `polyfit` for polynomials. But it works on models where there's no analytical solution — like neural nets.

That's Notebook 2.